# 05 - Silver : nettoyage des séismes

Dans cette étape je pars de la table bronze et je fabrique une table propre :
- je garde seulement les séismes (type earthquake), les autres événements sont hors périmètre
- je convertis les colonnes dans les bons types (nombres, dates)
- j'applique 4 règles de qualité, les lignes qui ne passent pas vont dans une table de rejets avec leur motif
- j'enlève les doublons (même id) en gardant la version la plus récente
- j'écris la table silver partitionnée par année
- je teste le MERGE en rechargeant une semaine déjà présente

## 1. J'ouvre la SparkSession et je définis les chemins des tables

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("05_silver").getOrCreate()

CHEMIN_BRONZE = "/lakehouse/bronze/seismes"
CHEMIN_SILVER = "/lakehouse/silver/seismes"
CHEMIN_REJETS = "/lakehouse/silver_rejets/seismes"

print("Application :", spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 15:48:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Application : 05_silver


## 2. Je lis la table bronze

In [3]:
bronze = spark.read.format("delta").load(CHEMIN_BRONZE)

nb_bronze = bronze.count()
print("Lignes dans bronze :", nb_bronze, "(attendu : 1604291, comme au notebook 04)")

26/11/12 15:49:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Lignes dans bronze : 1604291 (attendu : 1604291, comme au notebook 04)


## 3. Je sépare les séismes des autres événements (hors périmètre)
Les explosions, glissements de terrain, etc. ne sont pas des erreurs, ils ne font juste pas partie de mon sujet. Je les compte mais je ne les mets pas dans les rejets.
Dans bronze j'ai vu des types écrits avec des majuscules (Ice Quake, Landslide), donc je mets tout en minuscules avant de comparer.

In [4]:
type_propre = F.lower(F.trim(F.col("type")))

seismes_bruts = bronze.filter(type_propre == "earthquake")
hors_perimetre = bronze.filter((type_propre != "earthquake") | F.col("type").isNull())

nb_seismes_bruts = seismes_bruts.count()
nb_hors_perimetre = hors_perimetre.count()

print("Séismes             :", nb_seismes_bruts)
print("Hors périmètre      :", nb_hors_perimetre)
print("Vérification somme  :", nb_seismes_bruts + nb_hors_perimetre == nb_bronze)

Séismes             : 1567121
Hors périmètre      : 37170
Vérification somme  : True


## 4. Je convertis les colonnes dans les bons types
En bronze tout est en texte. Ici je transforme les dates en vraies dates et les nombres en nombres.
Si une valeur ne peut pas être convertie (texte vide, valeur bizarre), Spark met null : c'est ce qui me permet de repérer les lignes invalides juste après.

In [5]:
FORMAT_DATE = "yyyy-MM-dd'T'HH:mm:ss.SSSX"   # format des dates USGS, ex : 2019-07-05T00:00:20.000Z

types = (
    seismes_bruts
    .withColumn("date", F.to_timestamp("time", FORMAT_DATE))
    .withColumn("mis_a_jour", F.to_timestamp("updated", FORMAT_DATE))
    .withColumn("magnitude", F.col("mag").cast("double"))
    .withColumn("lat", F.col("latitude").cast("double"))
    .withColumn("lon", F.col("longitude").cast("double"))
    .withColumn("profondeur", F.col("depth").cast("double"))
)

# je compare le texte d'origine et la valeur convertie sur quelques lignes
types.select("time", "date", "mag", "magnitude", "depth", "profondeur").show(3, truncate=False)

+------------------------+----------------------+----------+----------+---------------+---------------+
|time                    |date                  |mag       |magnitude |depth          |profondeur     |
+------------------------+----------------------+----------+----------+---------------+---------------+
|2020-07-17T18:53:48.070Z|2020-07-17 18:53:48.07|1.84000003|1.84000003|32.169998168945|32.169998168945|
|2020-08-20T04:48:53.600Z|2020-08-20 04:48:53.6 |1.18      |1.18      |14.26          |14.26          |
|2018-08-10T23:57:33.730Z|2018-08-10 23:57:33.73|3.41      |3.41      |59             |59.0           |
+------------------------+----------------------+----------+----------+---------------+---------------+
only showing top 3 rows



## 5. J'applique les 4 règles de qualité
Chaque ligne reçoit un motif si elle ne respecte pas une règle, sinon le motif reste vide (null).
Si une ligne a plusieurs problèmes, je garde seulement le premier motif trouvé.
Je mets le résultat en cache car je m'en sers pour les rejets et pour les lignes valides.

Je ne remplace pas les magnitudes manquantes par la médiane : la magnitude est la valeur que j'étudie (Gutenberg-Richter, valeur b). Inventer une valeur créerait de faux séismes à la même magnitude et fausserait les résultats. Ces lignes sont très peu nombreuses par rapport au total, donc je les rejette, et je les garde dans silver_rejets pour pouvoir les compter.

In [6]:
avec_motif = types.withColumn(
    "motif",
    F.when(F.col("id").isNull() | (F.trim(F.col("id")) == ""), "id_manquant")
     .when(F.col("magnitude").isNull(), "magnitude_manquante")
     .when(
         F.col("lat").isNull() | F.col("lon").isNull()
         | (F.col("lat") < -90) | (F.col("lat") > 90)
         | (F.col("lon") < -180) | (F.col("lon") > 180),
         "coordonnees_invalides")
     .when(F.col("date").isNull(), "date_invalide")
).cache()

# null = la ligne est valide
avec_motif.groupBy("motif").count().withColumnRenamed("count", "nb_lignes").orderBy(F.desc("nb_lignes")).show()

+-------------------+---------+
|              motif|nb_lignes|
+-------------------+---------+
|               NULL|  1566506|
|magnitude_manquante|      615|
+-------------------+---------+



## 6. J'écris les lignes rejetées dans la table silver_rejets
Je garde les colonnes d'origine (en texte) pour pouvoir voir ce qui n'allait pas, plus le motif.

In [7]:
rejets = (
    avec_motif
    .filter(F.col("motif").isNotNull())
    .select("id", "time", "latitude", "longitude", "depth", "mag", "type",
            "motif", "fichier_source", "date_ingestion")
)

rejets.write.format("delta").mode("overwrite").save(CHEMIN_REJETS)

table_rejets = spark.read.format("delta").load(CHEMIN_REJETS)
nb_rejets = table_rejets.count()
print("Lignes rejetées :", nb_rejets)
table_rejets.select("id", "time", "mag", "motif", "fichier_source").show(5, truncate=False)

Lignes rejetées : 615


+----------+------------------------+----+-------------------+----------------------------------------------------------------+
|id        |time                    |mag |motif              |fichier_source                                                  |
+----------+------------------------+----+-------------------+----------------------------------------------------------------+
|nc73406936|2020-06-11T08:10:02.640Z|NULL|magnitude_manquante|hdfs://namenode:9000/landing/seismes/2020/seismes_2020-06-05.csv|
|av62194196|2020-07-09T07:37:01.000Z|NULL|magnitude_manquante|hdfs://namenode:9000/landing/seismes/2020/seismes_2020-07-03.csv|
|nc71109449|2018-08-10T14:13:07.080Z|NULL|magnitude_manquante|hdfs://namenode:9000/landing/seismes/2018/seismes_2018-08-10.csv|
|av62192591|2020-07-06T00:57:05.740Z|NULL|magnitude_manquante|hdfs://namenode:9000/landing/seismes/2020/seismes_2020-07-03.csv|
|nc73203736|2019-06-29T16:51:58.220Z|NULL|magnitude_manquante|hdfs://namenode:9000/landing/seismes/2019/

## 7. Je prépare les lignes valides et j'ajoute les colonnes dont j'aurai besoin
- annee et mois : pour les graphiques dans le temps et pour partitionner la table
- classe_profondeur : superficiel (moins de 70 km), intermediaire (70 à 300 km), profond (plus de 300 km)

Je renomme aussi les colonnes en français pour que ce soit plus clair.
Une profondeur un peu négative n'est pas une erreur (séisme au-dessus du niveau de la mer), je la garde.

In [8]:
valides = (
    avec_motif
    .filter(F.col("motif").isNull())
    .withColumn("annee", F.year("date"))
    .withColumn("mois", F.month("date"))
    .withColumn(
        "classe_profondeur",
        F.when(F.col("profondeur").isNull(), "inconnue")
         .when(F.col("profondeur") < 70, "superficiel")
         .when(F.col("profondeur") <= 300, "intermediaire")
         .otherwise("profond"))
    .select(
        "id", "date", "annee", "mois",
        F.col("lat").alias("latitude"),
        F.col("lon").alias("longitude"),
        "profondeur", "classe_profondeur", "magnitude",
        F.col("magType").alias("type_magnitude"),
        F.col("place").alias("lieu"),
        F.col("status").alias("statut"),
        F.col("net").alias("reseau"),
        "mis_a_jour", "fichier_source", "date_ingestion")
)

nb_valides = valides.count()
print("Lignes valides :", nb_valides)

Lignes valides : 1566506


## 8. J'enlève les doublons : pour chaque id je garde la version la plus récente
Un séisme peut apparaître 2 fois (par exemple à la limite entre 2 semaines, ou si je recharge un fichier).
Delta refuse un MERGE si plusieurs lignes ont le même id, donc je dois dédoublonner avant.
Je numérote les lignes de chaque id de la plus récente (mis_a_jour) à la plus ancienne, et je garde la numéro 1.

In [9]:
fenetre = Window.partitionBy("id").orderBy(F.desc("mis_a_jour"))

silver = (
    valides
    .withColumn("rang", F.row_number().over(fenetre))
    .filter(F.col("rang") == 1)
    .drop("rang")
)

nb_silver = silver.count()
nb_doublons = nb_valides - nb_silver
print("Lignes après dédoublonnage :", nb_silver)
print("Doublons supprimés         :", nb_doublons)

Lignes après dédoublonnage : 1566506
Doublons supprimés         : 0


## 9. J'écris la table silver, partitionnée par année
partitionBy("annee") crée un dossier par année dans HDFS : quand je filtre sur une année, Spark lit seulement ce dossier.
repartition("annee") avant l'écriture me donne un seul fichier par année (sinon Spark ferait plein de petits fichiers).
mode overwrite : si je relance la cellule, la table est remplacée.

In [10]:
(
    silver.repartition("annee")
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("annee")
    .save(CHEMIN_SILVER)
)
print("Table silver écrite dans", CHEMIN_SILVER)

Table silver écrite dans /lakehouse/silver/seismes


## 10. Je vérifie la table silver : lignes, années, profondeurs, magnitudes

In [11]:
table_silver = spark.read.format("delta").load(CHEMIN_SILVER)

print("Lignes dans silver          :", table_silver.count(), "(doit être égal à", nb_silver, ")")
print("Séismes de magnitude >= 4.5 :", table_silver.filter(F.col("magnitude") >= 4.5).count())
print("Profondeurs négatives (gardées) :", table_silver.filter(F.col("profondeur") < 0).count())

print("Séismes par année :")
table_silver.groupBy("annee").count().withColumnRenamed("count", "nb_seismes").orderBy("annee").show()

print("Séismes par classe de profondeur :")
table_silver.groupBy("classe_profondeur").count().withColumnRenamed("count", "nb_seismes").orderBy(F.desc("nb_seismes")).show()

table_silver.printSchema()

Lignes dans silver          : 1566506 (doit être égal à 1566506 )


Séismes de magnitude >= 4.5 : 74362


Profondeurs négatives (gardées) : 36463
Séismes par année :


+-----+----------+
|annee|nb_seismes|
+-----+----------+
| 2016|    122058|
| 2017|    130789|
| 2018|    187012|
| 2019|    184528|
| 2020|    198174|
| 2021|    159582|
| 2022|    146831|
| 2023|    147900|
| 2024|    152176|
| 2025|    137456|
+-----+----------+

Séismes par classe de profondeur :


+-----------------+----------+
|classe_profondeur|nb_seismes|
+-----------------+----------+
|      superficiel|   1425512|
|    intermediaire|    128356|
|          profond|     12638|
+-----------------+----------+

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- profondeur: double (nullable = true)
 |-- classe_profondeur: string (nullable = true)
 |-- magnitude: double (nullable = true)
 |-- type_magnitude: string (nullable = true)
 |-- lieu: string (nullable = true)
 |-- statut: string (nullable = true)
 |-- reseau: string (nullable = true)
 |-- mis_a_jour: timestamp (nullable = true)
 |-- fichier_source: string (nullable = true)
 |-- date_ingestion: timestamp (nullable = true)



## 11. Bilan qualité : je vérifie que chaque ligne de bronze est comptée quelque part
lignes de bronze = hors périmètre + rejetées + doublons supprimés + lignes dans silver

In [12]:
print("Lignes reçues (bronze)  :", nb_bronze)
print("Hors périmètre          :", nb_hors_perimetre)
print("Rejetées                :", nb_rejets)
print("Doublons supprimés      :", nb_doublons)
print("Gardées dans silver     :", nb_silver)
print("Aucune ligne perdue     :", nb_hors_perimetre + nb_rejets + nb_doublons + nb_silver == nb_bronze)

Lignes reçues (bronze)  : 1604291
Hors périmètre          : 37170
Rejetées                : 615
Doublons supprimés      : 0
Gardées dans silver     : 1566506
Aucune ligne perdue     : True


## 12. Test du MERGE : je recharge une semaine déjà présente
L'ingestion incrémentale n'est pas dans mon périmètre, donc je montre le MERGE en rechargeant la dernière semaine de 2025, qui est déjà dans silver.
Pour voir une mise à jour, je simule une révision de l'USGS sur un séisme : statut reviewed, magnitude + 0.1 et date de mise à jour récente.
Résultat attendu : aucune ligne ajoutée (pas de doublon) et une seule ligne modifiée.

In [13]:
# la semaine rechargée (je la reprends depuis le traitement, comme si le fichier arrivait de nouveau)
semaine = silver.filter(F.col("fichier_source").endswith("seismes_2025-12-26.csv"))

id_revise = semaine.orderBy("date").first()["id"]
print("Séismes dans la semaine rechargée :", semaine.count())
print("Séisme dont je simule la révision :", id_revise)

est_revise = F.col("id") == id_revise
recharge = (
    semaine
    .withColumn("magnitude", F.when(est_revise, F.round(F.col("magnitude") + 0.1, 2)).otherwise(F.col("magnitude")))
    .withColumn("statut", F.when(est_revise, F.lit("reviewed")).otherwise(F.col("statut")))
    .withColumn("mis_a_jour", F.when(est_revise, F.current_timestamp()).otherwise(F.col("mis_a_jour")))
)

print("Avant le MERGE :")
table_silver.filter(F.col("id") == id_revise).select("id", "magnitude", "statut", "mis_a_jour").show(truncate=False)

Séismes dans la semaine rechargée : 2218
Séisme dont je simule la révision : ak2025ziztas
Avant le MERGE :


+------------+---------+--------+-----------------------+
|id          |magnitude|statut  |mis_a_jour             |
+------------+---------+--------+-----------------------+
|ak2025ziztas|1.9      |reviewed|2026-03-04 04:56:05.214|
+------------+---------+--------+-----------------------+



## 13. Je fais le MERGE dans silver
- même id dans silver et dans la semaine rechargée : je mets à jour, mais seulement si la version rechargée est plus récente
- id qui n'existe pas encore : je l'ajoute

In [14]:
nb_avant = spark.read.format("delta").load(CHEMIN_SILVER).count()

(
    DeltaTable.forPath(spark, CHEMIN_SILVER).alias("cible")
    .merge(recharge.alias("source"), "cible.id = source.id")
    .whenMatchedUpdateAll(condition="source.mis_a_jour > cible.mis_a_jour")
    .whenNotMatchedInsertAll()
    .execute()
)

apres = spark.read.format("delta").load(CHEMIN_SILVER)
print("Lignes avant le MERGE :", nb_avant)
print("Lignes après le MERGE :", apres.count(), "(même nombre = aucun doublon)")

print("Après le MERGE :")
apres.filter(F.col("id") == id_revise).select("id", "magnitude", "statut", "mis_a_jour").show(truncate=False)

Lignes avant le MERGE : 1566506


Lignes après le MERGE : 1566506 (même nombre = aucun doublon)
Après le MERGE :


+------------+---------+--------+--------------------------+
|id          |magnitude|statut  |mis_a_jour                |
+------------+---------+--------+--------------------------+
|ak2025ziztas|2.0      |reviewed|2026-11-12 15:53:25.222631|
+------------+---------+--------+--------------------------+



## 14. L'historique de la table silver
Je vois l'écriture (WRITE) puis le MERGE, avec le nombre de lignes modifiées et ajoutées.

In [15]:
(
    DeltaTable.forPath(spark, CHEMIN_SILVER)
    .history()
    .select(
        "version", "timestamp", "operation",
        F.col("operationMetrics")["numOutputRows"].alias("lignes_ecrites"),
        F.col("operationMetrics")["numTargetRowsUpdated"].alias("lignes_modifiees"),
        F.col("operationMetrics")["numTargetRowsInserted"].alias("lignes_ajoutees"))
    .orderBy("version")
    .show(truncate=False)
)

+-------+-----------------------+---------+--------------+----------------+---------------+
|version|timestamp              |operation|lignes_ecrites|lignes_modifiees|lignes_ajoutees|
+-------+-----------------------+---------+--------------+----------------+---------------+
|0      |2026-11-12 15:52:32.561|WRITE    |1566506       |NULL            |NULL           |
|1      |2026-11-12 15:53:52.212|MERGE    |137456        |1               |0              |
+-------+-----------------------+---------+--------------+----------------+---------------+



## 15. Je libère le cache et je ferme la SparkSession

In [16]:
avec_motif.unpersist()
spark.stop()

26/11/12 15:54:05 WARN Dispatcher: Message RemoteProcessDisconnected(172.25.0.8:56882) dropped. Could not find BlockManagerMaster.
